## 06 — Model Training & Comparison

> **Training and comparative evaluation of Random Forest, XGBoost, and Gradient Boosting for NBA win prediction.** > Input: `data/3_features/features_nba_data_2000-01_2025-26.csv`
> Models saved in: `models/`

---

**Pipeline**

1. Setup & import from `src/`
2. Dataset Overview — split shapes, target distribution, feature list
3. Training — classification and regression
4. Classification Evaluation — comparative table, ROC curves, confusion matrix
5. Regression Evaluation — comparative table, scatter plots, residuals, win derivation
6. Feature Importance — top 20 by model, cross-model consistency
7. Error Analysis — misclassified games, segmentation by season / back-to-back / COVID

### 1. Setup & import from `src/`

In [ ]:
import logging
import shutil
import sys
import warnings
from pathlib import Path

# Set the project root directory and add it to the system path
ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
# Suppress warning messages for cleaner output
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
# Import metrics for evaluating model performance
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)

# Import custom project modules for loading, preprocessing, training, and evaluation
from src.ml.data_loader import load_splits, FEATURES_PATH
from src.ml.preprocessing import (
    FEATURE_COLS,
    HOME_FEATURES,
    AWAY_FEATURES,
    DIFF_FEATURES,
    CONTEXTUAL_FEATURES,
    transform,
)
from src.ml.trainer import train_all, MODELS_DIR
from src.ml.evaluator import plot_roc_curves, predict_proba
from src.loader import load_config

# Load config
config = load_config()

# Configure the logging format to track notebook execution progress
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger(__name__)

# Set global pandas display options for better readability
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)
# Apply visual theme to seaborn plots
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)

# Notebook configuration constants
NOTEBOOK_VERSION = "1.0.0"
MODEL_NAMES      = ["random_forest", "xgboost", "gradient_boosting"]
MODEL_LABELS     = ["Random Forest", "XGBoost", "Gradient Boosting"]
MODEL_COLORS     = ["steelblue", "darkorange", "forestgreen"]

# Log initial notebook configuration info
log.info("Notebook v%s — initialized", NOTEBOOK_VERSION)
log.info("Project root : %s", ROOT)
log.info("Models dir   : %s", MODELS_DIR)
log.info("Features path: %s", FEATURES_PATH)

### 2. Dataset Overview

#### 2.1 Split Shapes, Target Distribution, Feature List

In [ ]:
# --- Classification data splits ---
# Load training, validation, and testing sets for the classification task (predicting 'home_win')
(X_train_clf, y_train_clf), (X_val_clf, y_val_clf), (X_test_clf, y_test_clf) = load_splits("classification")

print("=" * 60)
print("Classification  (target: home_win)")
print("=" * 60)
# Output summary statistics for each classification split
for name, X, y in [("train", X_train_clf, y_train_clf),
                    ("val",   X_val_clf,   y_val_clf),
                    ("test",  X_test_clf,  y_test_clf)]:
    hw = y.mean()
    print(f"  {name:5s}  rows={len(y):>6d}  features={X.shape[1]:>3d}  "
          f"home_win_rate={hw:.3f}  away_win_rate={1-hw:.3f}")

# --- Regression data splits ---
# Load training, validation, and testing sets for the regression task (predicting 'point_differential')
(X_train_reg, y_train_reg), (X_val_reg, y_val_reg), (X_test_reg, y_test_reg) = load_splits("regression")

print()
print("=" * 60)
print("Regression  (target: point_differential)")
print("=" * 60)
# Output summary statistics for each regression split
for name, X, y in [("train", X_train_reg, y_train_reg),
                    ("val",   X_val_reg,   y_val_reg),
                    ("test",  X_test_reg,  y_test_reg)]:
    print(f"  {name:5s}  rows={len(y):>6d}  "
          f"pt_diff  mean={y.mean():+.2f}  std={y.std():.2f}  "
          f"[{y.min():.0f}, {y.max():.0f}]")

In [ ]:
# --- Canonical feature list ---
# Group features for systematic display and verification
feature_groups = {
    f"Home rolling/seasonal  ({len(HOME_FEATURES)})": HOME_FEATURES,
    f"Away rolling/seasonal  ({len(AWAY_FEATURES)})": AWAY_FEATURES,
    f"Differential (home−away)  ({len(DIFF_FEATURES)})": DIFF_FEATURES,
    f"Contextual  ({len(CONTEXTUAL_FEATURES)})": CONTEXTUAL_FEATURES,
}

# Iterate through groups and list all feature columns
for group_name, cols in feature_groups.items():
    print(f"\n{'─'*55}")
    print(f"  {group_name}")
    print(f"{'─'*55}")
    for col in cols:
        # Highlight the special status of the COVID bubble feature
        marker = "  <- COVID context" if col == "is_covid_bubble" else ""
        print(f"  {col}{marker}")

# Print final count of all features used in the model
print(f"\n{'═'*55}")
print(f"  Total features: {len(FEATURE_COLS)}")
print(f"{'═'*55}")
print()
# Note regarding the inclusion of the Orlando Bubble data
print("NOTE: 'is_covid_bubble' is included as a feature—")
print("      the model will learn its relevance automatically")
print("      without excluding the games from the Orlando Bubble.")

### 2.2 Baseline calculation

In [ ]:
# --- 1. Classification Baseline ---
# Determine the majority class from the training set to establish a baseline for classification
if hasattr(y_train_clf, "value_counts"):
    majority_class = y_train_clf.value_counts().idxmax()
else:
    majority_class = 1 if np.mean(y_train_clf) >= 0.5 else 0

# Calculate accuracy scores for the training, validation, and test sets
BASELINE_ACC      = np.mean(y_train_clf == majority_class)
baseline_val_acc  = np.mean(y_val_clf == majority_class)
baseline_test_acc = np.mean(y_test_clf == majority_class)

print("Classification Baseline (Majority Class Prediction)")
print("-" * 55)
print(f"Train Baseline Accuracy (Majority Class '{majority_class}'): {BASELINE_ACC:.4f}")
print(f"Validation Baseline Accuracy:                      {baseline_val_acc:.4f}")
print(f"Test Baseline Accuracy:                          {baseline_test_acc:.4f}")

In [ ]:
# --- 2. Regression Baseline ---
# Use the training target mean as a naive prediction for regression
train_mean = y_train_reg.mean()

# Create constant prediction arrays based on the training mean
y_pred_val_base  = np.full(len(y_val_reg), train_mean)
y_pred_test_base = np.full(len(y_test_reg), train_mean)

print("\nRegression Baseline (Train Mean Prediction)")
print("-" * 55)
print(f"Train Target Mean used for predictions:        {train_mean:+.2f}")
print(f"Validation Baseline MAE:                       {mean_absolute_error(y_val_reg, y_pred_val_base):.4f}")
print(f"Test Baseline MAE:                             {mean_absolute_error(y_test_reg, y_pred_test_base):.4f}")

### 3. Training

Each `train_all` call trains the three models (Random Forest, XGBoost, Gradient Boosting) and saves the model + the preprocessor in `models/`.

* `train_all("classification")` → target `home_win`
* `train_all("regression")` → target `point_differential`

In [ ]:
MODELS_DIR = ROOT / config["models"]["models_dir"]

# Remove the models directory if it exists to train the models from scratch
if MODELS_DIR.exists():
    shutil.rmtree(MODELS_DIR)

In [ ]:
# Log the start of the classification training process
log.info("=" * 50)
log.info("Training — classification (home_win)")
log.info("=" * 50)

# Execute the training pipeline for all models defined in the classification task
# timing_clf returns a DataFrame tracking the duration of each model's training
timing_clf = train_all("classification")

# Display the training duration for each model in a clean, formatted table
display(timing_clf.style.format({"train_time_s": "{:.2f}s"}).hide(axis="index"))

In [ ]:
# Log the start of the regression training process
log.info("=" * 50)
log.info("Training — regression (point_differential)")
log.info("=" * 50)

# Execute the training pipeline for all models defined in the regression task
# timing_reg returns a DataFrame tracking the training duration for each model
timing_reg = train_all("regression")

# Display the formatted training duration table for regression models
display(timing_reg.style.format({"train_time_s": "{:.2f}s"}).hide(axis="index"))

### 4. Classification Evaluation (`home_win`)

In [ ]:
def eval_clf_split(X_raw, y, task="classification"):
    """Returns a DataFrame with accuracy/F1/AUC metrics for a given split."""
    # Load the fitted preprocessor for the specific task
    prep = joblib.load(MODELS_DIR / f"preprocessor_{task}_v1.joblib")
    # Transform raw input features using the loaded preprocessor
    X = transform(X_raw, prep)
    
    rows = []
    # Iterate through model names to calculate evaluation metrics
    for name in MODEL_NAMES:
        model = joblib.load(MODELS_DIR / f"{name}_{task}_v1.joblib")
        y_pred = model.predict(X)
        y_prob = predict_proba(model, X)
        
        # Append performance metrics for each model
        rows.append({
            "model":    name,
            "accuracy": accuracy_score(y, y_pred),
            "f1":       f1_score(y, y_pred, zero_division=0),
            "auc_roc":  roc_auc_score(y, y_prob),
        })
    return pd.DataFrame(rows).set_index("model")


# Evaluate models on both validation and test sets
val_clf  = eval_clf_split(X_val_clf,  y_val_clf)
test_clf = eval_clf_split(X_test_clf, y_test_clf)

# Combine results into a comparison DataFrame
compare_clf = pd.concat(
    [val_clf.add_suffix("_val"), test_clf.add_suffix("_test")],
    axis=1
)[["accuracy_val", "accuracy_test", "f1_val", "f1_test", "auc_roc_val", "auc_roc_test"]]

# Calculate delta (test - validation) to identify performance drift
compare_clf["Δacc"]     = compare_clf["accuracy_test"] - compare_clf["accuracy_val"]
compare_clf["Δauc_roc"] = compare_clf["auc_roc_test"]  - compare_clf["auc_roc_val"]

# Display the comparison table with highlights
print(f"Baseline (majority class): {BASELINE_ACC:.3f}\n")
display(
    compare_clf.style
    .format("{:.4f}")
    .highlight_max(subset=["accuracy_val", "accuracy_test", "f1_val", "f1_test",
                            "auc_roc_val", "auc_roc_test"], color="#d4edda")
    .highlight_min(subset=["Δacc", "Δauc_roc"], color="#f8d7da")
    .set_caption("Classification — validation vs. test comparison (Δ = test − val)")
)

print("\nInterpretation: Δ << 0 indicates overfitting.")

In [ ]:
# Generate and display the ROC curves for all models evaluated on the test set
# The plot_roc_curves function handles the computation of TPR/FPR and plotting logic
fig = plot_roc_curves("classification")

# Add a title and adjust layout for better visual presentation
fig.suptitle("ROC Curves — Test Set", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Load the preprocessor and transform the test set for consistent evaluation across all models
prep = joblib.load(MODELS_DIR / "preprocessor_classification_v1.joblib")
X_test_t = transform(X_test_clf, prep)

# Create a side-by-side subplot figure for comparing confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Iterate through models to generate and plot each confusion matrix
for ax, name, label in zip(axes, MODEL_NAMES, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    y_pred = model.predict(X_test_t)
    cm = confusion_matrix(y_test_clf, y_pred)

    disp = ConfusionMatrixDisplay(cm, display_labels=["Away Win", "Home Win"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(f"{label}\nAccuracy={accuracy_score(y_test_clf, y_pred):.3f}")

# Finalize plot layout
plt.suptitle("Confusion Matrix — Test Set", fontsize=13)
plt.tight_layout()
plt.show()

# Calculate and print specific diagnostic metrics (Precision/Recall) for each model
print("Metrics derived from confusion matrices:\n")
for name, label in zip(MODEL_NAMES, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    y_pred = model.predict(X_test_t)
    tn, fp, fn, tp = confusion_matrix(y_test_clf, y_pred).ravel()
    
    # Precision: How many of the predicted Home Wins were correct?
    # Recall: How many of the actual Home Wins did the model capture?
    precision_home = tp / (tp + fp)
    recall_home = tp / (tp + fn)

    print(f"{label:<18} TN={tn:>4d}  FP={fp:>4d}  FN={fn:>4d}  TP={tp:>4d}  "
          f"Precision home={precision_home:.4f}  Recall home={recall_home:.4f}")

### 5. Regression Evaluation (`point_differential`)

Metrics reported on **validation** and **test**.
The `home_win` prediction is also derived from the sign of `point_differential_pred > 0` and compared with the accuracy of the direct classification model.

In [ ]:
def eval_reg_split(X_raw, y, task="regression"):
    """Returns a DataFrame with MAE, RMSE, and R² metrics for a given split."""
    # Load the fitted preprocessor for the regression task
    prep = joblib.load(MODELS_DIR / f"preprocessor_{task}_v1.joblib")
    # Transform raw input features
    X = transform(X_raw, prep)
    
    rows = []
    # Calculate performance metrics for each model
    for name in MODEL_NAMES:
        model = joblib.load(MODELS_DIR / f"{name}_{task}_v1.joblib")
        y_pred = model.predict(X)
        rmse = float(np.sqrt(mean_squared_error(y, y_pred)))
        rows.append({
            "model": name,
            "mae":   mean_absolute_error(y, y_pred),
            "rmse":  rmse,
            "r2":    r2_score(y, y_pred),
        })
    return pd.DataFrame(rows).set_index("model")


# Perform evaluations for ML models
val_reg = eval_reg_split(X_val_reg, y_val_reg)
test_reg = eval_reg_split(X_test_reg, y_test_reg)

# Build baseline metrics directly using the global arrays computed in Section 2
baseline_reg = pd.DataFrame(
    {
        "mae_val": [mean_absolute_error(y_val_reg, y_pred_val_base)],
        "mae_test": [mean_absolute_error(y_test_reg, y_pred_test_base)],
        "rmse_val": [float(np.sqrt(mean_squared_error(y_val_reg, y_pred_val_base)))],
        "rmse_test": [float(np.sqrt(mean_squared_error(y_test_reg, y_pred_test_base)))],
        "r2_val": [r2_score(y_val_reg, y_pred_val_base)],
        "r2_test": [r2_score(y_test_reg, y_pred_test_base)],
    },
    index=["baseline_mean_train"],
)

# Consolidate results for comparison
compare_reg = pd.concat(
    [val_reg.add_suffix("_val"), test_reg.add_suffix("_test")],
    axis=1
)[["mae_val", "mae_test", "rmse_val", "rmse_test", "r2_val", "r2_test"]]

compare_reg = pd.concat([compare_reg, baseline_reg])

# Calculate performance deltas (test - validation)
compare_reg["Δmae"] = compare_reg["mae_test"]  - compare_reg["mae_val"]
compare_reg["Δr2"]  = compare_reg["r2_test"]   - compare_reg["r2_val"]

# Display the comparison table
display(
    compare_reg.style
    .format("{:.4f}")
    .highlight_min(subset=["mae_val", "mae_test", "rmse_val", "rmse_test"], color="#d4edda")
    .highlight_max(subset=["r2_val",  "r2_test"],  color="#d4edda")
    .highlight_max(subset=["Δmae"],  color="#f8d7da")
    .set_caption("Regression — validation vs. test comparison with baseline (Δ = test − val)")
)

In [ ]:
# Load the regression preprocessor and transform the test set for consistent evaluation
prep_reg = joblib.load(MODELS_DIR / "preprocessor_regression_v1.joblib")
X_test_reg_t = transform(X_test_reg, prep_reg)

# Create a subplot figure for side-by-side comparison of regression performance
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

# Iterate through models to visualize predicted versus actual values
for ax, name, color, label in zip(axes, MODEL_NAMES, MODEL_COLORS, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")
    y_pred = model.predict(X_test_reg_t)
    r2 = r2_score(y_test_reg, y_pred)
    mae = mean_absolute_error(y_test_reg, y_pred)

    # Plot actual vs predicted values with transparency for density
    ax.scatter(y_test_reg, y_pred, alpha=0.25, s=8, color=color, rasterized=True)
    
    # Draw a diagonal reference line representing perfect predictions
    lim = max(abs(y_test_reg.min()), abs(y_test_reg.max())) * 1.05
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1, label="Perfect")
    
    # Configure axes limits and labels for readability
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel("Actual point_differential")
    ax.set_ylabel("Predicted" if ax is axes[0] else "")
    ax.set_title(f"{label}\nR²={r2:.3f}  MAE={mae:.2f}")
    ax.set_aspect("equal")

# Set the overall figure title and adjust layout
fig.suptitle("Scatter Predicted vs Actual — Test Set (Regression)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Create a 2x3 grid for residual analysis (histograms and Q-Q plots)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for col, name, color, label in zip(range(3), MODEL_NAMES, MODEL_COLORS, MODEL_LABELS):
    # Load the trained model and calculate residuals on the test set
    model = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")
    y_pred = model.predict(X_test_reg_t)
    residuals = y_pred - y_test_reg.values

    # Top row: Histogram of residuals vs. theoretical normal distribution
    ax_hist = axes[0][col]
    ax_hist.hist(residuals, bins=60, density=True, color=color, alpha=0.7, edgecolor="none")
    mu, sigma = residuals.mean(), residuals.std()
    x_range = np.linspace(residuals.min(), residuals.max(), 300)
    ax_hist.plot(x_range, stats.norm.pdf(x_range, mu, sigma), "k-", lw=1.5, label=f"N({mu:.1f},{sigma:.1f})")
    ax_hist.axvline(0, color="red", lw=1.2, linestyle="--")
    ax_hist.set_title(f"{label}\nresidual — mean={mu:+.2f}  std={sigma:.2f}")
    ax_hist.set_xlabel("Residual (pred − actual)")
    ax_hist.legend(fontsize=8)

    # Bottom row: Q-Q plot to assess normality of residuals
    ax_qq = axes[1][col]
    (osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist="norm")
    ax_qq.scatter(osm, osr, alpha=0.3, s=6, color=color, rasterized=True)
    ax_qq.plot(osm, slope * np.array(osm) + intercept, "k-", lw=1.5)
    ax_qq.set_title(f"Q-Q  (R={r:.3f})")
    ax_qq.set_xlabel("Theoretical Quantiles")
    ax_qq.set_ylabel("Sample Quantiles")

# Set figure title and finalize layout
fig.suptitle("Residual Distribution — Test Set (Regression)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Compare the accuracy of deriving a "win" from regression predictions versus direct classification

print("Derived accuracy (reg pred > 0 ↦ home_win=1)  vs.  direct classification\n")
print(f"{'Model':<22} {'Derived Acc':>14} {'Direct Clf Acc':>16}")
print("-" * 56)

# Load the preprocessor for the classification task
prep_clf = joblib.load(MODELS_DIR / "preprocessor_classification_v1.joblib")
X_test_clf_t = transform(X_test_clf, prep_clf)

for name, label in zip(MODEL_NAMES, MODEL_LABELS):
    # Derived from Regression: Predict win if point differential > 0
    model_reg   = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")
    y_pred_reg  = model_reg.predict(X_test_reg_t)
    y_win_from_reg = (y_pred_reg > 0).astype(int)
    acc_derived = accuracy_score(y_test_clf, y_win_from_reg)

    # Direct Classification: Predict win using the classification model
    model_clf   = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    y_pred_clf  = model_clf.predict(X_test_clf_t)
    acc_direct  = accuracy_score(y_test_clf, y_pred_clf)

    # Calculate difference between approaches
    delta = acc_direct - acc_derived
    print(f"{label:<22}  {acc_derived:>12.4f}  {acc_direct:>14.4f}   Δ={delta:+.4f}")

# Compare against the baseline
print(f"{'Baseline (majority)':<22}  {BASELINE_ACC:>12.3f}")
print("\n→ If Δ > 0, direct classification is preferable to derived regression.")

### 6. Feature Importance

Top 20 features for each classification model.
Cross-model comparison: are the most important features consistent across RF, XGBoost, and GBM?

#### 6.1 Classification - Feature Importance

In [ ]:
# Create a figure for side-by-side comparison of Top-20 feature importance
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

importance_dfs = {}

# Iterate through each classification model to extract and plot feature importance
for ax, name, color, label in zip(axes, MODEL_NAMES, MODEL_COLORS, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    prep  = joblib.load(MODELS_DIR / "preprocessor_classification_v1.joblib")
    feat_names = prep.feature_cols

    # Extract feature importance values and identify the top 20
    importances = model.feature_importances_
    top20_idx   = np.argsort(importances)[::-1][:20]
    top20_names = [feat_names[i] for i in top20_idx]
    top20_vals  = importances[top20_idx]

    # Store full importance mapping for further analysis
    importance_dfs[name] = pd.Series(importances, index=feat_names, name=name).sort_values(ascending=False)

    # Horizontal bar plot for the top 20 features
    ax.barh(range(20), top20_vals[::-1], color=color, alpha=0.85)
    ax.set_yticks(range(20))
    ax.set_yticklabels(top20_names[::-1], fontsize=8)
    ax.set_xlabel("Importance")
    ax.set_title(label, fontsize=11)
    # Align visually to show highest importance closest to the right
    ax.invert_xaxis()

fig.suptitle("Top-20 Feature Importance — Classification", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Create a DataFrame combining all model importances
all_imp = pd.DataFrame(importance_dfs)
all_imp.columns = MODEL_LABELS

# Calculate the rank for each feature per model (1 = most important)
# A lower rank indicates higher relative importance
rank_df = all_imp.rank(ascending=False, method="min").astype(int)
rank_df["mean_rank"] = rank_df.mean(axis=1)
rank_df = rank_df.sort_values("mean_rank")

print("Top-15 features by average rank across the three classification models\n")
# Display the ranked features with a color gradient highlighting top performers
display(
    rank_df.head(15).style
    .background_gradient(cmap="YlOrRd_r", subset=MODEL_LABELS, vmin=1, vmax=30)
    .format("{:.1f}", subset=["mean_rank"])
    .set_caption("Rank 1 = most important feature in the model")
)

# Check the importance rank of 'is_covid_bubble' specifically
if "is_covid_bubble" in rank_df.index:
    covid_row = rank_df.loc[["is_covid_bubble"]]
    print(f"\nis_covid_bubble — rank across the three models:")
    display(covid_row.style.format("{:.0f}", subset=MODEL_LABELS).format("{:.1f}", subset=["mean_rank"]))
else:
    print("\nis_covid_bubble is not present in the transformed feature set.")

#### 6.2 Regression - Feature Importance

In [ ]:
# Create a figure for side-by-side comparison of Top-20 feature importance for Regression
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

importance_reg_dfs = {}
prep_reg_imp = joblib.load(MODELS_DIR / "preprocessor_regression_v1.joblib")
feat_names_reg = prep_reg_imp.feature_cols

# Iterate through regression models to visualize Top-20 feature importance
for ax, name, color, label in zip(axes, MODEL_NAMES, MODEL_COLORS, MODEL_LABELS):
    model = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")

    importances = model.feature_importances_
    top20_idx   = np.argsort(importances)[::-1][:20]
    top20_names = [feat_names_reg[i] for i in top20_idx]
    top20_vals  = importances[top20_idx]

    importance_reg_dfs[name] = pd.Series(importances, index=feat_names_reg, name=name).sort_values(ascending=False)

    ax.barh(range(20), top20_vals[::-1], color=color, alpha=0.85)
    ax.set_yticks(range(20))
    ax.set_yticklabels(top20_names[::-1], fontsize=8)
    ax.set_xlabel("Importance")
    ax.set_title(label, fontsize=11)
    ax.invert_xaxis()

fig.suptitle("Top-20 Feature Importance — Regression", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate and rank feature importance for Regression
all_imp_reg = pd.DataFrame(importance_reg_dfs)
all_imp_reg.columns = MODEL_LABELS

rank_reg_df = all_imp_reg.rank(ascending=False, method="min").astype(int)
rank_reg_df["mean_rank"] = rank_reg_df.mean(axis=1)
rank_reg_df = rank_reg_df.sort_values("mean_rank")

print("Top-15 features by average rank across the three regression models\n")
display(
    rank_reg_df.head(15).style
    .background_gradient(cmap="YlOrRd_r", subset=MODEL_LABELS, vmin=1, vmax=30)
    .format("{:.1f}", subset=["mean_rank"])
    .set_caption("Rank 1 = most important feature in the model")
)

#### 6.3 Classification vs Regression

In [ ]:
# Compare feature rankings between Classification and Regression tasks
rank_task_compare = pd.DataFrame(
    {
        "rank_classification": rank_df["mean_rank"],
        "rank_regression": rank_reg_df["mean_rank"],
    }
).dropna()

# Calculate the delta between task ranks to see which features shift importance
rank_task_compare["Δrank_reg_minus_clf"] = (
    rank_task_compare["rank_regression"] - rank_task_compare["rank_classification"]
)
rank_task_compare["abs_Δrank"] = rank_task_compare["Δrank_reg_minus_clf"].abs()
rank_task_compare = rank_task_compare.sort_values("abs_Δrank", ascending=False)

print("Top-20 features with largest average rank difference between classification and regression\n")
display(
    rank_task_compare.head(20).style
    .format("{:.1f}")
    .background_gradient(cmap="RdYlGn_r", subset=["rank_classification", "rank_regression"], vmin=1, vmax=len(FEATURE_COLS))
    .background_gradient(cmap="Blues", subset=["abs_Δrank"])
    .set_caption("Δ > 0 = more important in classification; Δ < 0 = more important in regression")
)

### 7. Error Analysis

* **7a** — Games misclassified by all three models: common patterns.
* **7b** — Regression: games with very high absolute error (historical upsets or noise?).
* **7c** — Error segmentation by season, back-to-back, COVID bubble.

In [ ]:
# Reload the full CSV to include metadata (season, date, B2B, COVID, etc.) for result analysis
full_df   = pd.read_csv(FEATURES_PATH)
test_meta = full_df[full_df["split"] == "test"].copy().reset_index(drop=True)

# Generate predictions for all three classifiers on the test set
pred_clf = {}
prob_clf = {}
for name in MODEL_NAMES:
    model = joblib.load(MODELS_DIR / f"{name}_classification_v1.joblib")
    pred_clf[name] = model.predict(X_test_clf_t)
    prob_clf[name] = predict_proba(model, X_test_clf_t)

# Generate predictions for all three regressors on the test set
pred_reg = {}
for name in MODEL_NAMES:
    model = joblib.load(MODELS_DIR / f"{name}_regression_v1.joblib")
    pred_reg[name] = model.predict(X_test_reg_t)

# Consolidate results into a unified DataFrame for inspection
test_results = test_meta[[
    "game_date", "season",
    "home_team_abbreviation", "away_team_abbreviation",
    "home_wl", "away_wl", "home_pts", "away_pts",
    "is_covid_bubble",
    "home_is_back_to_back", "away_is_back_to_back",
]].copy()

# Add ground truth and actual point spread
test_results["y_true_clf"]   = y_test_clf.values
test_results["y_true_reg"]   = y_test_reg.values
test_results["actual_spread"] = test_results["home_pts"] - test_results["away_pts"]

# Map predictions to the results DataFrame using simplified model keys
for name in MODEL_NAMES:
    short = name.replace("random_forest", "rf").replace("gradient_boosting", "gbm").replace("xgboost", "xgb")
    test_results[f"pred_clf_{short}"] = pred_clf[name]
    test_results[f"pred_reg_{short}"] = pred_reg[name]

# Identify erroneous predictions for each classifier
wrong_rf  = test_results["pred_clf_rf"]  != test_results["y_true_clf"]
wrong_xgb = test_results["pred_clf_xgb"] != test_results["y_true_clf"]
wrong_gbm = test_results["pred_clf_gbm"] != test_results["y_true_clf"]

# Aggregate error counts
test_results["all_wrong"] = wrong_rf & wrong_xgb & wrong_gbm
test_results["n_wrong"]   = wrong_rf.astype(int) + wrong_xgb.astype(int) + wrong_gbm.astype(int)

# Display error summary
n_all_wrong = test_results["all_wrong"].sum()
print(f"Total games in test set     : {len(test_results):>6}")
print(f"Missed by all three models  : {n_all_wrong:>6}  ({n_all_wrong/len(test_results)*100:.1f}%)")
print(f"Missed by at least one model: {(test_results['n_wrong']>0).sum():>6}")

In [ ]:
# Analyze characteristics of games misclassified by all three models
misclassified = test_results[test_results["all_wrong"]].copy()
correct = test_results[~test_results["all_wrong"]].copy()

print("=" * 60)
print("Games misclassified by RF + XGBoost + GBM")
print("=" * 60)
print(f"Distribution of y_true (home_win):")
print(misclassified["y_true_clf"].value_counts().rename({0: "Away Win", 1: "Home Win"}).to_string())

print(f"\nActual absolute spread (margin of victory):")
print(f"  Misclassified: mean={misclassified['actual_spread'].abs().mean():.2f}  "
      f"median={misclassified['actual_spread'].abs().median():.2f}")
print(f"  Correct      : mean={correct['actual_spread'].abs().mean():.2f}  "
      f"median={correct['actual_spread'].abs().median():.2f}")

print("\nNote: Misclassified games tend to have tighter margins, indicating genuine upsets.")

# Top 10 most "surprising" games (smallest spread among misclassified)
print("\nTop 10 upsets (smallest absolute spread) among games misclassified by all models:")
cols_show = ["game_date", "season", "home_team_abbreviation", "away_team_abbreviation",
             "home_pts", "away_pts", "actual_spread", "y_true_clf"]
display(
    misclassified.sort_values("actual_spread", key=abs).head(10)[cols_show]
    .rename(columns={"y_true_clf": "home_win", "actual_spread": "spread"})
    .style.hide(axis="index")
)

# Histogram comparison of spreads
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(correct["actual_spread"].abs(),       bins=40, alpha=0.6, label="Correct",  color="steelblue", density=True)
ax.hist(misclassified["actual_spread"].abs(), bins=40, alpha=0.6, label="Wrong (all 3)", color="tomato",     density=True)
ax.set_xlabel("|Spread| actual (points)")
ax.set_ylabel("Density")
ax.set_title("Spread distribution — correct vs. misclassified games")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Calculate the absolute error (AE) for each regression model
for name in MODEL_NAMES:
    short = name.replace("random_forest", "rf").replace("gradient_boosting", "gbm").replace("xgboost", "xgb")
    test_results[f"ae_{short}"] = (test_results[f"pred_reg_{short}"] - test_results["y_true_reg"]).abs()

# Compute the mean absolute error across the three models
test_results["mean_ae"] = test_results[["ae_rf", "ae_xgb", "ae_gbm"]].mean(axis=1)

# Define and identify games where models struggled significantly (Blowout Threshold)
BLOWOUT_THRESHOLD = 20  
large_errors = test_results[test_results["mean_ae"] >= BLOWOUT_THRESHOLD].copy()
print(f"NBA blowout threshold for mean MAE: {BLOWOUT_THRESHOLD:.0f} points")
print(f"Games with high error (>= {BLOWOUT_THRESHOLD:.0f} points): {len(large_errors)} ({len(large_errors)/len(test_results)*100:.1f}%)\n")

# Display the top 10 games with the largest mean regression error
cols_reg = ["game_date", "season", "home_team_abbreviation", "away_team_abbreviation",
            "actual_spread", "pred_reg_rf", "pred_reg_xgb", "pred_reg_gbm", "mean_ae"]
print("Top 10 games with the highest mean regression error:")
display(
    large_errors.nlargest(10, "mean_ae")[cols_reg]
    .round(1).style.hide(axis="index")
    .set_caption("Actual spread vs. model predictions")
)

print("\nNote: Historically anomalous margins (blowouts or massive upsets) contribute to")
print("      the tail of the error distribution. Check for potential injuries or schedule-related rest.")

In [ ]:
# Create a figure for segmented error analysis: Seasonality and Back-to-Back schedules
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# ── Seasonal Error Analysis ──────────────────────────────────────────────
ax = axes[0]
err_by_season = (
    test_results.groupby("season")["all_wrong"]
    .mean()
    .reset_index()
    .rename(columns={"all_wrong": "err_rate_all3"})
)
ax.bar(range(len(err_by_season)), err_by_season["err_rate_all3"], color="steelblue", alpha=0.8)
ax.set_xticks(range(len(err_by_season)))
ax.set_xticklabels(err_by_season["season"].str[-5:], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Error Rate (all three models)")
ax.set_title("Errors by Season")

# ── Back-to-Back (B2B) Schedule Impact ───────────────────────────────────
ax = axes[1]
test_results["b2b_flag"] = (
    test_results["home_is_back_to_back"].astype(bool) |
    test_results["away_is_back_to_back"].astype(bool)
)
b2b_err = test_results.groupby("b2b_flag")["all_wrong"].mean().reset_index()
b2b_err["label"] = b2b_err["b2b_flag"].map({False: "No B2B", True: "At least 1 B2B"})
ax.bar(b2b_err["label"], b2b_err["all_wrong"], color=["steelblue", "darkorange"], alpha=0.85)
ax.set_ylabel("Error Rate (all three models)")
ax.set_title("Errors: Back-to-Back Impact")

# Add text labels on the bars for clarity
for i, (_, row) in enumerate(b2b_err.iterrows()):
    ax.text(i, row["all_wrong"] + 0.002, f"{row['all_wrong']:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

# Print summary table
print("Average error rate per segment:\n")
print(f"{'Segment':<30} {'Error Rate (all 3)':>22}")
print("-" * 56)
for _, row in b2b_err.iterrows():
    print(f"{row['label']:<30}  {row['all_wrong']:>20.3f}")